# Minimal repro: Unsloth DPO + Gemma 4 vision hangs at tokenization

**Symptom:** `DPOTrainer.train()` hangs silently at `Tokenizing 0/52 [00:00<?, ? examples/s]` with both `dataset_num_proc=1` and `dataset_num_proc=2`. CPU at ~0%, GPU at 0%, no error, no progress.

**Setup:** Gemma 4 E4B 4-bit, vision layers unfrozen, LoRA r=16, fresh SFT-warmstart adapter (or fresh LoRA — either reproduces). Synthetic preference pairs: 1024x1024 RGB images + short prompts/chosen/rejected.

**Diagnostics that work (function-level):**
- `processor(images=..., text=...)` direct call: returns in 0.1s
- `_UnslothDPOTrainer.process_row(features=pairs[0], ...)` direct call: returns in 0s
- Manual Python loop calling `process_row` over all pairs: returns in ~2s

**What hangs:** `dataset.map(process_row)` inside `DPOTrainer._prepare_dataset` — the same function works fine outside `dataset.map`.

**Workaround attempts that didn't help:**
- Setting `HF_DATASETS_CACHE` to a known-writable path on persistent volume
- Switching `dataset_num_proc` between 1 and 2
- Using `Features({"images": [HFImage()], ...})` for explicit Arrow type
- Pre-tokenizing manually and passing tokenized dataset (TRL still calls `dataset.map(..., remove_columns=["chosen","rejected"])` and errors because those columns are gone)

**Environment:** Modal A100 (40GB), Linux, Python 3.12.


## 1. Install

In [1]:

# Step 1: install unsloth-zoo first; let it resolve and possibly downgrade
%uv pip install unsloth-zoo --system

# Step 2: re-pin to versions from original bug report
%uv pip install pillow==11.3.0 --system
%uv pip install "transformers==5.6.2" --system
%uv pip install "trl==0.22.2" --system
%uv pip install "peft==0.19.1" --system
%uv pip install "datasets==4.3.0" --system

# Step 3: apply datta0's fix
%uv pip uninstall unsloth --system
%uv pip install "git+https://github.com/datta0/unsloth@dpo_mp_hang" --no-deps --system
!rm -rf unsloth_compiled_cache

Using Python 3.12.6 environment at: /usr/local
Resolved 85 packages in 67ms
Uninstalled 1 package in 129ms
Installed 1 package in 91ms
 - transformers==5.6.2
 + transformers==5.5.0
Note: you may need to restart the kernel to use updated packages.
Using Python 3.12.6 environment at: /usr/local
Audited 1 package in 6ms
Note: you may need to restart the kernel to use updated packages.
Using Python 3.12.6 environment at: /usr/local
Resolved 27 packages in 29ms
Uninstalled 1 package in 111ms
Installed 1 package in 96ms
 - transformers==5.5.0
 + transformers==5.6.2
Note: you may need to restart the kernel to use updated packages.
Using Python 3.12.6 environment at: /usr/local
Audited 1 package in 15ms
Note: you may need to restart the kernel to use updated packages.
Using Python 3.12.6 environment at: /usr/local
Audited 1 package in 11ms
Note: you may need to restart the kernel to use updated packages.
Using Python 3.12.6 environment at: /usr/local
Audited 1 package in 12ms
Note: you may nee

In [2]:

%uv pip install bitsandbytes --system

Using Python 3.12.6 environment at: /usr/local
Audited 1 package in 11ms
Note: you may need to restart the kernel to use updated packages.


In [3]:
import unsloth
import importlib.metadata as md
for pkg in ["bitsandbytes", "unsloth", "unsloth_zoo", "trl", "transformers", "datasets", "peft", "torch", "pillow"]:
    try:
        print(f"{pkg}: {md.version(pkg)}")
    except md.PackageNotFoundError:
        print(f"{pkg}: NOT INSTALLED")

print(f"\nunsloth location: {unsloth.__file__}")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!
bitsandbytes: 0.49.2
unsloth: 2026.4.8
unsloth_zoo: 2026.4.9
trl: 0.22.2
transformers: 5.6.2
datasets: 4.3.0
peft: 0.19.1
torch: 2.8.0+cu129
pillow: 11.3.0

unsloth location: /usr/local/lib/python3.12/site-packages/unsloth/__init__.py


## 2. Imports

In [4]:
from unsloth import FastVisionModel       # MUST be first
from trl import DPOTrainer, DPOConfig
from datasets import Dataset
from PIL import Image
import torch
import time
import os

## 3. Load Gemma 4 E4B 4-bit + fresh LoRA adapter

Use Unsloth's `FastVisionModel.get_peft_model` (NOT `PeftModel.from_pretrained`, which fails on `Gemma4ClippableLinear` module resolution).

In [5]:
model, tokenizer = FastVisionModel.from_pretrained(
    "unsloth/gemma-4-e4b-it",      
    load_in_4bit=True,
    use_gradient_checkpointing="unsloth",
)
print(f"Base loaded. GPU: {torch.cuda.memory_allocated()/1e9:.1f} GB")

model = FastVisionModel.get_peft_model(
    model,
    r=16,
    lora_alpha=32,
    lora_dropout=0,
    bias="none",
    finetune_vision_layers=True,
    finetune_language_layers=True,
    finetune_attention_modules=True,
    finetune_mlp_modules=True,
    target_modules="all-linear",
)
print("Trainable params:")
model.print_trainable_parameters()

==((====))==  Unsloth 2026.4.8: Fast Gemma4 patching. Transformers: 5.6.2.
   \\   /|    NVIDIA A100 80GB PCIe. Num GPUs = 1. Max memory: 79.251 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu129. CUDA: 8.0. CUDA Toolkit: 12.9. Triton: 3.4.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/2076 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/208 [00:00<?, ?B/s]

processor_config.json: 0.00B [00:00, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/32.2M [00:00<?, ?B/s]

Base loaded. GPU: 10.8 GB
Trainable params:
trainable params: 39,403,520 || all params: 7,980,504,352 || trainable%: 0.4937


## 4. Synthetic preference pairs (52 examples)

Just blank 1024x1024 RGB PIL images + short text. Real images and real preferences are not needed to reproduce — the hang is structural, not data-dependent.

Schema: flat `{images: [PIL], prompt: str, chosen: str, rejected: str}` — this is what Unsloth's vision DPO `process_row` expects (it does NOT want chat-template content blocks).

In [6]:
def make_pair(idx: int):
    img = Image.new("RGB", (1024, 1024), color=(idx % 256, 128, 200))
    return {
        "images": [img],
        "prompt": "Describe the image in one sentence.",
        "chosen": f"Image {idx} shows a solid color block.",
        "rejected": f"Image {idx} contains nothing useful at all.",
    }

pairs = [make_pair(i) for i in range(52)]
print(f"Built {len(pairs)} synthetic pairs")
print(f"Sample image: {pairs[0]['images'][0].size}")

Built 52 synthetic pairs
Sample image: (1024, 1024)


## 5. Diagnostic: confirm `process_row` works on its own

This call IS the function that `dataset.map` will invoke 52 times during DPOTrainer prep. It works fine in isolation.

In [7]:
from unsloth_compiled_cache.UnslothDPOTrainer import _UnslothDPOTrainer

start = time.time()
out = _UnslothDPOTrainer.process_row(
    features=pairs[0],
    processing_class=tokenizer,
    max_prompt_length=1024,
    max_completion_length=None,
    add_special_tokens=False,
)
print(f"Single call OK in {time.time()-start:.2f}s. Keys: {list(out.keys())}")

Single call OK in 0.03s. Keys: ['prompt_input_ids', 'pixel_values', 'chosen_input_ids', 'rejected_input_ids']


In [8]:
# Manual loop: call process_row on every pair. Should finish in ~2s.
start = time.time()
processed = []
for p in pairs:
    out = _UnslothDPOTrainer.process_row(
        features=p,
        processing_class=tokenizer,
        max_prompt_length=1024,
        max_completion_length=None,
        add_special_tokens=False,
    )
    processed.append(out)
print(f"Manual loop over {len(pairs)} pairs: {time.time()-start:.2f}s")

Manual loop over 52 pairs: 0.62s


## 6. Repro: with fix applied, tokenization succeeds but training fails at collator

**As originally filed (before Datta0's fix in install cell 2):** the trainer's `_prepare_dataset` calls `dataset.map(process_row, ...)` internally, which hung indefinitely at `0/52 [00:00<?, ? examples/s]` with CPU at ~0% and GPU at 0%, no error.

**With Datta0's fix applied (`git+https://github.com/datta0/unsloth@dpo_mp_hang`, see cell 2):** tokenization completes in ~6 seconds at ~32 examples/s. The trainer then fails at the data collator before any training step executes, with:

```
ValueError: You should supply an encoding or a list of encodings to this method
that includes input_ids, but you provided ['images', 'prompt', 'prompt_input_ids',
'pixel_values', 'chosen_input_ids', 'rejected_input_ids']
```

This is the same structural blocker reported in [issue #4174](https://github.com/unslothai/unsloth/issues/4174) (open since 2026-03-06, no upstream response). The default data collator does not understand the multimodal column set that the DPO data preparation produces; pre-tokenizing manually and stripping `chosen`/`rejected` runs into TRL's own `remove_unused_columns` step.

So Vision DPO via Unsloth currently has a two-step blocker history:
1. Tokenization hang in `dataset.map` - **fixed** in `datta0/unsloth@dpo_mp_hang` (issue #5196)
2. Multimodal collator missing `input_ids` field - **still open** (issue #4174)


In [9]:
pairs_ds = Dataset.from_list(pairs)
print(f"Dataset: {len(pairs_ds)} examples, columns: {pairs_ds.column_names}")

dpo_trainer = DPOTrainer(
    model=model,
    ref_model=None,
    processing_class=tokenizer,
    train_dataset=pairs_ds,
    args=DPOConfig(
        beta=0.1,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=4,
        num_train_epochs=3,
        learning_rate=5e-6,
        output_dir="/tmp/dpo-repro",
        max_length=2048,
        max_prompt_length=1024,
        remove_unused_columns=False,
        logging_steps=1,
        dataset_num_proc=1,    # also tried 2 — both hang
    ),
)

# Hangs here ↓
dpo_trainer.train()

Dataset: 52 examples, columns: ['images', 'prompt', 'chosen', 'rejected']


Extracting prompt in train dataset (num_proc=1):   0%|          | 0/52 [00:00<?, ? examples/s]

Applying chat template to train dataset (num_proc=1):   0%|          | 0/52 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/52 [00:00<?, ? examples/s]

[accelerate.utils.other|WARNING]Detected kernel version 4.4.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.
[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': 2}.
[transformers] ==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 52 | Num Epochs = 3 | Total steps = 39
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 4 x 1) = 4
 "-____-"     Trainable parameters = 39,403,520 of 7,980,504,352 (0.49% trained)


ValueError: You should supply an encoding or a list of encodings to this method that includes input_ids, but you provided ['images', 'prompt', 'prompt_input_ids', 'pixel_values', 'chosen_input_ids', 'rejected_input_ids']

## 7. Mitigation attempts (historical, before the fix)

The following were tried before Datta0's fix landed. They are kept here as a record of the pre-fix debugging path, not as currently-relevant workarounds.

1. **`dataset_num_proc=1` and `=2`** - both hung. Single-process meant no fork issue, but still wedged. *(No longer relevant: the fix addresses this code path.)*
2. **Forced `HF_DATASETS_CACHE` to a known-writable persistent path** - no change. *(No longer relevant.)*
3. **Explicit `Features({"images": [HFImage()], "prompt": Value("string"), ...})` schema** - Dataset constructed in 0s, but trainer still hung at the same step. *(No longer relevant.)*
4. **Pre-tokenize via manual loop then pass tokenized dataset** (with `prompt_input_ids`, `pixel_values`, `chosen_input_ids`, `rejected_input_ids` columns) - TRL's `_prepare_dataset` still calls `dataset.map(..., remove_columns=["chosen", "rejected"])` which then errors because those columns are gone. *(STILL RELEVANT post-fix: this remains the same workaround dead-end against the collator bug in #4174.)*

## Status as of 2026-04-27

- Tokenization hang reported in [#5196](https://github.com/unslothai/unsloth/issues/5196). Maintainer Datta0 responded within ~2 hours with the fix branch above. Fix validated empirically: `0/52` indefinite hang -> 6 seconds at 32 examples/s.
- Collator failure exposed by the fix matches [#4174](https://github.com/unslothai/unsloth/issues/4174). No upstream response on #4174 since March.
- For the CivicInsight v1 (May 15 submission), DPO is parked. The agentic verification layer addresses the residual failure modes that DPO would have targeted.
